[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/planessoria-ui/ModelVinya/blob/main/auto_labeling/train_yolo.ipynb)

# 🍇 ModelVinya · Entrenar YOLOv11 per comptar baies

Entrena un detector de baies amb les etiquetes que has fet a l'anotador.

### Abans de començar
1. **GPU:** *Entorno de ejecución → Cambiar tipo → GPU*.
2. A l'anotador (https://planessoria-ui.github.io/ModelVinya/), exporta **🗂 YOLO dataset (.zip)** amb totes les fotos revisades.
3. Executa les cel·les en ordre i puja el `.zip` quan et demani.

> Com més fotos etiquetades (idealment 100+), millor generalitzarà el model.

In [ ]:
# 1) Instal·lar ultralytics (YOLOv11)
!pip -q install ultralytics
import ultralytics; ultralytics.checks()

In [ ]:
# 2) Pujar el .zip exportat de l'anotador i preparar train/val (80/20)
from google.colab import files
up = files.upload()                      # selecciona el YOLO dataset .zip

import zipfile, glob, os, random, shutil
zname = list(up.keys())[0]
shutil.rmtree('/content/raw', ignore_errors=True); shutil.rmtree('/content/ds', ignore_errors=True)
zipfile.ZipFile(zname).extractall('/content/raw')

imgs = [p for p in glob.glob('/content/raw/**/*.*', recursive=True)
        if '/images/' in p.replace('\\', '/') and p.lower().endswith(('.jpg','.jpeg','.png'))]
random.seed(0); random.shuffle(imgs)
n = max(1, int(len(imgs) * 0.8))
splits = {'train': imgs[:n], 'val': imgs[n:] or imgs[:1]}
for split, subset in splits.items():
    for ip in subset:
        lp = ip.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
        os.makedirs(f'/content/ds/images/{split}', exist_ok=True)
        os.makedirs(f'/content/ds/labels/{split}', exist_ok=True)
        shutil.copy(ip, f'/content/ds/images/{split}/')
        if os.path.exists(lp): shutil.copy(lp, f'/content/ds/labels/{split}/')

with open('/content/ds/data.yaml', 'w') as f:
    f.write("path: /content/ds\ntrain: images/train\nval: images/val\nnc: 1\nnames: [baya]\n")
print(f"train: {len(splits['train'])} imatges | val: {len(splits['val'])} imatges")

In [ ]:
# 3) Entrenar (imgsz gran perquè les baies són petites en fotos grans)
!yolo detect train model=yolo11n.pt data=/content/ds/data.yaml epochs=100 imgsz=1280 batch=8 patience=30
# El millor model queda a: runs/detect/train/weights/best.pt

In [ ]:
# 4) Provar el model en una foto nova i descarregar el pes entrenat
from ultralytics import YOLO
from google.colab import files
model = YOLO('runs/detect/train/weights/best.pt')

test = files.upload()                    # puja una foto per provar
for name in test:
    r = model.predict(name, save=True, conf=0.25, imgsz=1280)[0]
    print(f"{name}: {len(r.boxes)} baies detectades")
# la imatge amb les caixes queda a runs/detect/predict/
files.download('runs/detect/train/weights/best.pt')   # guarda el teu model

## Notes
- `best.pt` és el teu **model entrenat**: detecta i compta baies en fotos noves.
- Per **més precisió**: més fotos etiquetades i variades (llum, varietat, fons), i pots pujar `epochs` o el model (`yolo11s.pt`, `yolo11m.pt`).
- El **diàmetre** no surt de YOLO: es calcula amb el radi dels cercles + l'**escala** (regle). Per això el **CSV** de l'anotador serveix per al model de calibre/pes.
- Per a **camp** (racims a la planta) pots entrenar igual amb les fotos de camp etiquetades.